# TRACER Getting Started

This notebook demonstrates how to apply TRACER to a custom text classification dataset.

Steps covered:

1. Create teacher traces
2. Generate embeddings
3. Train a routing policy
4. Evaluate coverage
5. Load and use a router
6. Use fallback LLM routing
7. Update the router with new traces

## Install Dependencies

```bash
pip install tracer-llm[embeddings]

In [3]:
import pandas as pd
import tracer

## Example Dataset

For demonstration purposes we create a small dataset containing text and labels.

In [4]:
df = pd.DataFrame(
    {
        "text": [
            "What is my account balance?",
            "Transfer money to John",
            "Show recent transactions",
            "Reset my debit card PIN",
            "Check my savings account balance",
            "Send $100 to Alice",
            "Show my last five transactions",
            "Change my card PIN",
            "How much money do I have?",
            "Transfer funds to Bob",
            "View transaction history",
            "Update debit card security code",
            "Check available balance",
            "Move money to another account",
            "Display recent purchases",
            "Reset banking password",
            "Show account statement",
            "Transfer funds internationally",
            "View account details",
            "Change account settings"
        ],
        "label": [
            "balance",
            "transfer",
            "transactions",
            "pin_reset",
            "balance",
            "transfer",
            "transactions",
            "pin_reset",
            "balance",
            "transfer",
            "transactions",
            "pin_reset",
            "balance",
            "transfer",
            "transactions",
            "pin_reset",
            "balance",
            "transfer",
            "transactions",
            "pin_reset"
        ]
    }
)

In [5]:
import json

with open("traces.jsonl", "w") as f:
    for _, row in df.iterrows():
        trace = {
            "input": row["text"],
            "teacher": row["label"]
        }
        f.write(json.dumps(trace) + "\n")

print("traces.jsonl created")

traces.jsonl created


In [6]:
texts = df["text"].tolist()

X = tracer.embed(texts)

print(X.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.42it/s]

(20, 384)


In [7]:
result = tracer.fit(
    "traces.jsonl",
    embeddings=X,
    config=tracer.FitConfig(
        target_teacher_agreement=0.95
    )
)

result

[tracer.fit +   0.0s] fit_frontier: X=(20, 384) targets=[0.85, 0.9, 0.95] max_fit_labels=8000
[tracer.fit +   0.0s] fit_frontier: split -> 12 train / 4 val / 4 cal
[tracer.fit +   0.0s]   build_global: surrogate sweep on 12 train / 4 val
[tracer.fit +   0.1s]     [  0.0s]  logreg_c1     f1=0.667
[tracer.fit +   0.1s]     [  0.0s]  logreg_c10    f1=0.667
[tracer.fit +   0.1s]     [  0.0s]  sgd_log       f1=0.667
[tracer.fit +   0.2s]     [  0.1s]  mlp_1h        f1=1.000
[tracer.fit +   0.3s]     [  0.1s]  mlp_2h        f1=0.250
[tracer.fit +   0.3s]     [  0.0s]  dt            f1=0.417
[tracer.fit +   0.8s]     [  0.5s]  rf            f1=0.375
[tracer.fit +   1.3s]     [  0.5s]  et            f1=0.667
[tracer.fit +   2.5s]     [  1.2s]  gbt           f1=0.375
[tracer.fit +   2.5s]   build_global: surrogate done in 2.5s  best=mlp_1h f1=1.000
[tracer.fit +   2.5s] build_l2d: target_TA=0.85
[tracer.fit +   2.5s]   stage_1: surrogate sweep on 12 train / 4 val
[tracer.fit +   2.5s]     [  0.

FitResult(artifact_dir='.tracer', manifest=ArtifactManifest(version='0.1.0', n_traces=20, label_space=['balance', 'pin_reset', 'transactions', 'transfer'], selected_method='global', target_teacher_agreement=0.95, coverage_cal=1.0, teacher_agreement_cal=1.0, embedding_dim=384, n_retrains=1, pipeline_path='.tracer\\pipeline.joblib', index_path='.tracer\\index', config_path='.tracer\\config.json', qualitative_report_path='.tracer\\qualitative_report.json'), qualitative_report=QualitativeReport(summary='Handled 20/20 (100.0%) by surrogate; deferred 0/20 (0.0%).', coverage=1.0, teacher_agreement_handled=0.95, slices=[SliceInsight(slice_name='length:short', predicate='length_bin == short', count=7, handled_rate=1.0, deferred_rate=0.0, teacher_agreement_handled=None, dominant_teacher_label='transfer'), SliceInsight(slice_name='length:medium', predicate='length_bin == medium', count=7, handled_rate=1.0, deferred_rate=0.0, teacher_agreement_handled=None, dominant_teacher_label='transactions'), 

## Understanding the Results

TRACER trains several surrogate models and selects the best one that satisfies the target teacher agreement.

Important metrics:

- Coverage
- Teacher Agreement
- Selected Method

Higher coverage means fewer LLM calls.

In [8]:
print(result)

FitResult(artifact_dir='.tracer', manifest=ArtifactManifest(version='0.1.0', n_traces=20, label_space=['balance', 'pin_reset', 'transactions', 'transfer'], selected_method='global', target_teacher_agreement=0.95, coverage_cal=1.0, teacher_agreement_cal=1.0, embedding_dim=384, n_retrains=1, pipeline_path='.tracer\\pipeline.joblib', index_path='.tracer\\index', config_path='.tracer\\config.json', qualitative_report_path='.tracer\\qualitative_report.json'), qualitative_report=QualitativeReport(summary='Handled 20/20 (100.0%) by surrogate; deferred 0/20 (0.0%).', coverage=1.0, teacher_agreement_handled=0.95, slices=[SliceInsight(slice_name='length:short', predicate='length_bin == short', count=7, handled_rate=1.0, deferred_rate=0.0, teacher_agreement_handled=None, dominant_teacher_label='transfer'), SliceInsight(slice_name='length:medium', predicate='length_bin == medium', count=7, handled_rate=1.0, deferred_rate=0.0, teacher_agreement_handled=None, dominant_teacher_label='transactions'), 

In [9]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\HP\tracer\notebooks
['.tracer', '01-quickstart.ipynb', '02-static-tracer.ipynb', '03-dynamic-tracer.ipynb', '04-getting-started.ipynb', 'traces.jsonl']


In [10]:
import os

print(os.path.exists(".tracer"))

if os.path.exists(".tracer"):
    print(os.listdir(".tracer"))

True
['all_traces.jsonl', 'config.json', 'frontier.json', 'index.embeddings.npy', 'manifest.json', 'pipeline.joblib', 'qualitative_report.json']


In [11]:
router = tracer.load_router(".tracer")
print("Router loaded successfully")

Router loaded successfully


## Loading the Router

After training, the router can be loaded from the generated `.tracer` artifact directory and used for inference.

In [12]:
from tracer import Embedder

embedder = Embedder.from_sentence_transformers(
    "BAAI/bge-small-en-v1.5"
)

router = tracer.load_router(
    ".tracer",
    embedder=embedder
)

In [13]:
router.predict(
    "What is my account balance?"
)

{'label': 'transfer', 'decision': 'handled', 'accept_score': 1.0, 'stage': 0}

## Router Decisions

The router decides whether the surrogate model can confidently answer a request or whether it should be deferred to the teacher LLM.

In [14]:
def fallback():
    return "fallback_response"

router.predict(
    "Very unusual banking request",
    fallback=fallback
)

{'label': 'transfer', 'decision': 'handled', 'accept_score': 1.0, 'stage': 0}

## LLM Fallback

If the surrogate is not confident enough, TRACER can defer the request to a fallback LLM.

In [15]:
import os

os.listdir(".tracer")

['all_traces.jsonl',
 'config.json',
 'frontier.json',
 'index.embeddings.npy',
 'manifest.json',
 'pipeline.joblib',
 'qualitative_report.json']

## Generated Artifacts

Training creates a `.tracer` directory containing:

- manifest.json
- pipeline.joblib
- frontier.json
- config.json

These files are used during inference.

## Continual Learning

As new production traces are collected, TRACER can be retrained to improve coverage while maintaining teacher agreement.

In [16]:
# Example only

# tracer.update(
#     "new_traces.jsonl",
#     embeddings=X_new
# )

## Cost Savings

Without TRACER:
- Every request goes to an LLM.

With TRACER:
- Easy requests are handled locally.
- Difficult requests are deferred.

Higher coverage directly reduces LLM costs.